**DATA INGESTION PIPELINE**

- Data loading from llamaindex simple directory reader
- Data normaliation for text cleaning
- Data chunking and embedding using llmaindex ingestion pipeline
- Create Milvus database schema and load entities to vector database with indexing

In [1]:
#import libraries

import re
import unicodedata

from dotenv import load_dotenv
from llama_index.core import Document, SimpleDirectoryReader
from llama_index.core.extractors import TitleExtractor
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.readers.file import PyMuPDFReader
from pymilvus import DataType, MilvusClient

load_dotenv()

True

In [2]:
#load pdf files from given path

file_extractor = {".pdf": PyMuPDFReader()}
reader = SimpleDirectoryReader(input_files=["../data/Analysis of the Effectiveness of ARIMA, SARIMA, and SVR.pdf"], file_extractor=file_extractor)
document = reader.load_data()

In [3]:
# data normalization/ cleaning

class DataNormalizer:
    """Normalizes text of Documents produced by SimpleDirectoryReader."""

    def __init__(self, documents: list[Document]):
        self.documents = documents

    def normalize(self) -> list[Document]:
        for doc in self.documents:
            doc.set_content(self._normalize_text(doc.text))
        return self.documents

    def _normalize_text(self, text: str) -> str:
        text = unicodedata.normalize("NFKC", text)  # convert characters to standard equivalents
        text = self._fix_hyphenation(text)  # hyphened words take into one word
        text = self._collapse_newlines(text) # replace new line with space
        text = self._collapse_whitespace(text) # handle repeated spaces or tabs
        return text.strip()

    def _fix_hyphenation(self, text: str) -> str:
        return re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    def _collapse_newlines(self, text: str) -> str:
        text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
        text = re.sub(r"\n{2,}", "\n\n", text)
        return text

    def _collapse_whitespace(self, text: str) -> str:
        return re.sub(r"[ \t]{2,}", " ", text)

In [4]:
#create normalized data

data_normalizer = DataNormalizer(documents=document)
normalized_text = data_normalizer.normalize()

In [5]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/home/nipuna/Documents/Personal/ininsight/ResearchLens/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-15 14:32:00,455 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-09-15 14:32:00,507 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-09-15 14:32:00,864 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redire

In [10]:
# create the pipeline with transformations
# chunking, titleextraction and embedding
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=150, chunk_overlap=10),
        TitleExtractor(),
        embed_model,
        # OpenAIEmbedding()
    ]
)

# run the pipeline
nodes = []
for doc in range(len(normalized_text)):

    text = normalized_text[doc]
    node = pipeline.run(documents=[text])
    nodes.extend(node)


  0%|          | 0/1 [00:00<?, ?it/s]2026-09-15 14:34:08,053 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-15 14:34:09,577 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-15 14:34:10,562 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-15 14:34:13,469 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-15 14:34:14,495 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-15 14:34:15,312 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
  0%|          | 0/1 [00:00<?, ?it/s]2026-09-15 14:34:15,591 - INFO - Retrying request to /chat/completions in 0.421066 seconds
2026-09-15 14:34:17,176 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-15 14:34:18,589 - INFO - HTTP Reques

In [11]:
nodes

[TextNode(id_='052a6618-b447-433c-b0f8-1a487ca0e344', embedding=[-0.08417539298534393, 0.04051797091960907, 0.004585843067616224, 0.0702212005853653, 0.035556018352508545, -0.023907089605927467, -0.04342348501086235, 0.03530963137745857, 0.044656407088041306, -0.02453421801328659, -0.0420059859752655, 0.0972953587770462, -0.045255888253450394, -0.04604130610823631, -0.03522464632987976, 0.024110639467835426, -0.11257270723581314, -0.05993278697133064, 0.0011497167870402336, -0.056531041860580444, -0.03362854942679405, 0.0044687711633741856, -0.01783318631350994, -0.02893258072435856, 0.10530702769756317, 0.038832083344459534, -0.10710939764976501, 0.009900880977511406, -0.03921664133667946, 0.08757239580154419, -0.011542466469109058, 0.06473611295223236, 0.03778596967458725, -0.004452107474207878, -0.057504862546920776, 0.08613766729831696, 0.03096010908484459, 0.049328748136758804, -0.006756196264177561, 0.08589591085910797, -0.01775168627500534, -0.042389314621686935, -0.016377389430

In [12]:
print("The context length of embeddings:", len(nodes[3].embedding))

The context length of embeddings: 384


In [13]:
# create database client

client = MilvusClient(
    uri="http://localhost:19530",
    token="root:Milvus",
)

In [21]:
collection_name = "paper_chunks_2"


schema = client.create_schema(
    auto_id=False,
    enable_dynamic_field=False,
)


schema.add_field(
    field_name="id",
    datatype=DataType.VARCHAR,
    max_length=100,
    is_primary=True,
)

schema.add_field(
    field_name="title",
    datatype=DataType.VARCHAR,
    max_length=500,
)

schema.add_field(
    field_name="page",
    datatype=DataType.INT64,
)

schema.add_field(
    field_name="chunk_index",
    datatype=DataType.INT64,
)


schema.add_field(
    field_name="text",
    datatype=DataType.VARCHAR,
    max_length=10000,
)

schema.add_field(
    field_name="embedding",
    datatype=DataType.FLOAT_VECTOR,
    dim=384,
)

{'auto_id': False, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 100}, 'is_primary': True, 'auto_id': False}, {'name': 'title', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 500}}, {'name': 'page', 'description': '', 'type': <DataType.INT64: 5>}, {'name': 'chunk_index', 'description': '', 'type': <DataType.INT64: 5>}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 10000}}, {'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 384}}], 'enable_dynamic_field': False, 'enable_namespace': False}

In [22]:
# define indexing parameters

index_params = client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    index_type="AUTOINDEX",
    metric_type="COSINE",
)

In [23]:
# create collection with indexing strategy

client.create_collection(
    collection_name=collection_name,
    schema=schema,
    index_params=index_params,
)

In [25]:
# load entities to database

chunk_id=0
for node in nodes:

        data = [{
                "id": node.id_,

                "title": node.metadata["document_title"],

                "page": int(node.metadata["source"]),
                
                "chunk_index": chunk_id,

                "text": node.text,

                "embedding": node.embedding,
        }]
        result = client.insert(
                collection_name="paper_chunks_2",
                data=data
        )
        chunk_id = chunk_id+1
        print(result)

{'insert_count': 1, 'ids': ['052a6618-b447-433c-b0f8-1a487ca0e344']}
{'insert_count': 1, 'ids': ['5b939d08-326e-48cf-9111-0834d9eaa096']}
{'insert_count': 1, 'ids': ['b749b0fd-245f-4b7d-a1a0-b7a6ea5ce2b7']}
{'insert_count': 1, 'ids': ['d1c17b54-82a2-48f2-acdd-9fccdeb22452']}
{'insert_count': 1, 'ids': ['9f84534c-794d-4d85-a4d7-b0af9a851e35']}
{'insert_count': 1, 'ids': ['9c3c0b4b-955d-4f15-8ed7-0e98c3a5361b']}
{'insert_count': 1, 'ids': ['be73e569-f3cf-473e-8123-acd94a4fc3b8']}
{'insert_count': 1, 'ids': ['2c8ecb8e-487e-4d72-98d8-933a1821ed26']}
{'insert_count': 1, 'ids': ['c1c62d2d-8e63-4efc-9ca8-0605a0c9e831']}
{'insert_count': 1, 'ids': ['aa2f54ec-8c90-4c8b-9395-6669a7096806']}
{'insert_count': 1, 'ids': ['969358ef-4632-4a88-ac21-9ca59f27e608']}
{'insert_count': 1, 'ids': ['cac95edc-f2df-4b7e-9bab-5dd03cd712b4']}
{'insert_count': 1, 'ids': ['ad667189-120d-4d48-bd38-a38939eda00e']}
{'insert_count': 1, 'ids': ['568ea7ac-cdb3-4169-8d60-169e75f72377']}
{'insert_count': 1, 'ids': ['384c4

In [26]:
print("Exists:", client.has_collection(collection_name))

count = client.query(
    collection_name=collection_name,
    filter="",
    output_fields=["count(*)"],
)

print("Entity count:", count)

Exists: True
Entity count: data: ["{'count(*)': 134}"], extra_info: {}


In [27]:
# test loaded data

results = client.query(
    collection_name=collection_name,
    filter="",
    output_fields=[
        "id",
        "title",
        "page",
        "chunk_index",
        "page",
        "text",
        "embedding",
    ],
    limit=3,
)

for row in results:
    print(row)
    print("-" * 80)

{'title': 'Advances in Time Series Analysis and Forecasting: A Comprehensive Review and Application of Support Vector Methods', 'page': 18, 'chunk_index': 131, 'text': 'In Advances in Neural Information Processing Systems 9; MIT Press: Cambridge, MA, USA, 1996. 32. Ding, S.; Hua, X.; Yu, J. An overview on nonparallel hyperplane support vector machine algorithms. Neural Comput. Appl. 2014, 25, 975–982. [CrossRef] 33. Shcherbakov, M.V.; Brebels, A.; Shcherbakova, N.L.; Tyukov, A.P.', 'embedding': [-0.07855717092752457, -0.05482318997383118, 0.05694279074668884, -0.025685135275125504, 0.046471014618873596, 0.05823472887277603, -0.008865214884281158, -0.009482198394834995, -0.01617187075316906, -0.00690658250823617, -0.09905605763196945, 0.09085405617952347, -0.06896618753671646, -0.05389620363712311, -0.1232551857829094, 0.01056644693017006, -0.04207393899559975, -0.08056177943944931, 0.018950004130601883, 0.039868757128715515, -0.013689927756786346, 0.04724366217851639, -0.03043325990438

In [28]:
# test query - semantic search against the indexed collection

# embed_model = OpenAIEmbedding()

query_text = "How accurate is SARIMA compared to ARIMA for wind farm energy forecasting?"
query_embedding = embed_model.get_query_embedding(query_text)

search_results = client.search(
    collection_name=collection_name,
    data=[query_embedding],
    limit=5,
    output_fields=["title", "page", "chunk_index", "text"],
)

for hit in search_results[0]:
    print(f"score: {hit['distance']:.4f} | page: {hit['entity']['page']} | chunk: {hit['entity']['chunk_index']}")
    print(hit["entity"]["text"])
    print("-" * 80)

score: 0.7867 | page: 12 | chunk: 78
1] 0.1 - 0.6 0 3.4. Monthly Forecasts The forecast results were monthly predictions made using three tested methods for each of the wind farms. For both ARIMA and SARIMA methods, tests were conducted with the same parameter ranges of p, d, q (test 1). For both farms, all methods were tested, and predictions were made for 12 months ahead.
--------------------------------------------------------------------------------
score: 0.7763 | page: 1 | chunk: 1
Energies 2024, 17, 4803. https://doi.org/ 10.3390/en17194803 Academic Editor: John Boland Received: 12 August 2024 Revised: 17 September 2024 Accepted: 20 September 2024 Published: 25 September 2024 Copyright: © 2024 by the authors. Licensee MDPI, Basel, Switzerland.
--------------------------------------------------------------------------------
score: 0.7652 | page: 1 | chunk: 4
); mazur@prz.edu.pl (D.M.); gregor@prz.edu.pl (G.D.) 2 Faculty of Electrical Engineering, Bialystok University of Technolog